## now IPCC analytical CO2 IRF benchmark
 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

In [2]:
# --- IPCC analytical CO2 impulse response ("remaining fraction") ---
import numpy as np

# AR6-style coefficients (sum ~ 1)
IPCC_A = np.array([0.2173, 0.2240, 0.2824, 0.2763])  # a0,a1,a2,a3
IPCC_ALPHA_CO2 = np.array([0.0, 394.4, 36.54, 4.304])  # years; index 0 placeholder

# H grid for IPCC curve (0..200 years)
H_ipcc = np.arange(0, 201, 1)

# pIRF_IPCC(H) = a0 + sum_{i=1..3} a_i exp(-H/alpha_i)
pIRF_ipcc = (
    IPCC_A[0]
    + IPCC_A[1] * np.exp(-H_ipcc / IPCC_ALPHA_CO2[1])
    + IPCC_A[2] * np.exp(-H_ipcc / IPCC_ALPHA_CO2[2])
    + IPCC_A[3] * np.exp(-H_ipcc / IPCC_ALPHA_CO2[3])
)

# If you need list form (e.g., exporting)
pIRF_ipcc_list = pIRF_ipcc.tolist()
pIRF_ipcc_list

[1.0,
 0.9345251433514736,
 0.881133921191247,
 0.8373585015922008,
 0.8012416932667857,
 0.7712311040344035,
 0.7460952421568477,
 0.7248570112558832,
 0.7067409927073082,
 0.6911316570412732,
 0.677540238510535,
 0.6655784767528133,
 0.6549378018439959,
 0.6453728342077808,
 0.6366883048203195,
 0.6287286866129532,
 0.6213699749892455,
 0.6145131719061268,
 0.6080791203422738,
 0.6020044091988496,
 0.5962381267190023,
 0.5907392865203889,
 0.5854747868042868,
 0.5804177922131702,
 0.5755464507235245,
 0.5708428761247556,
 0.566292341033272,
 0.5618826368039314,
 0.5576035657479487,
 0.5534465382376206,
 0.5494042529627097,
 0.5454704431092888,
 0.5416396748036167,
 0.5379071869948702,
 0.5342687641948385,
 0.5307206352716697,
 0.5272593929049253,
 0.5238819294270121,
 0.5205853856621213,
 0.5173671100761811,
 0.5142246261080782,
 0.5111556059937422,
 0.5081578497445239,
 0.5052292682186109,
 0.502367869444057,
 0.4995717475262475,
 0.4968390736107659,
 0.49416808848211885,
 0.4915570

In [3]:
paths_my = {
    "2030": "pIRF/pirf_median_by_ssp-all_H200_MY2030.xlsx",
    "2040": "pIRF/pirf_median_by_ssp-all_H200_MY2040.xlsx",
    "2050": "pIRF/pirf_median_by_ssp-all_H200_MY2050.xlsx",
}

paths_est = {
    "CO2": "to_comp_withWatanabe_paper/pIRF_CO2.xlsx",
    "CH4": "to_comp_withWatanabe_paper/pIRF_CH4.xlsx",
    "N2O": "to_comp_withWatanabe_paper/pIRF_N2O.xlsx",
}

mapping = [
    ("RCP2.6", "ssp126"),
    ("RCP4.5", "ssp245"),
    ("RCP6.0", "ssp460"),
    ("RCP8.5", "ssp585"),
]

def detect_cols(df):
    df.columns = [str(c) for c in df.columns]
    col_gas = [c for c in df.columns if c.lower() == "gas"][0]
    col_ssp = [c for c in df.columns if "ssp" in c.lower() or "scenario" in c.lower()][0]
    col_h = [c for c in df.columns if c.lower() in ["h","horizon","time","t","year"]][0]
    col_val = [c for c in df.columns if c.lower() in ["value","median","pirf"]][0]
    return col_gas, col_ssp, col_h, col_val

# Load MY data
my_data = {}
for year, path in paths_my.items():
    df = pd.read_excel(path, sheet_name="median_long")
    my_data[year] = dict(zip(
        ["df","gas","ssp","h","val"],
        (df, *detect_cols(df))
    ))


In [4]:
out_dir = "pIRF/plot_compare_Watanabe_AND_IPCC"
os.makedirs(out_dir, exist_ok=True)

panel_paths = []

for gas in ["CO2","CH4","N2O"]:
    for rcp, ssp in mapping:
        
        plt.figure()
        
        # ----- FaIR lines -----
        if gas == "CO2":
            styles = {"2030":"-","2040":"--","2050":":"}
            for year in ["2030","2040","2050"]:
                df = my_data[year]["df"]
                gcol = my_data[year]["gas"]
                scol = my_data[year]["ssp"]
                hcol = my_data[year]["h"]
                vcol = my_data[year]["val"]
                
                df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
                df_sel = df_sel.sort_values(hcol)
                
                plt.plot(
                    df_sel[hcol], df_sel[vcol],
                    linestyle=styles[year],
                    color="lightblue",
                    label=f"FaIR - MY{year}, {ssp}"
                )
        else:
            df = my_data["2030"]["df"]
            gcol = my_data["2030"]["gas"]
            scol = my_data["2030"]["ssp"]
            hcol = my_data["2030"]["h"]
            vcol = my_data["2030"]["val"]
            
            df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
            df_sel = df_sel.sort_values(hcol)
            
            plt.plot(
                df_sel[hcol], df_sel[vcol],
                linestyle="-",
                color="lightblue",
                label=f"FaIR - MY2030, {ssp}"
            )
        # ----- IPCC analytical decay (same curve for all SSPs) -----
        if gas == "CO2":
            plt.plot(
                H_ipcc, pIRF_ipcc,
                color="grey",
                linestyle="-",
                label="IPCC analytical IRF (AR6)"
            )


        
        # ----- Watanabe & Cherubini (2026) -----
        df_est = pd.read_excel(paths_est[gas])
        df_est.columns = [str(c) for c in df_est.columns]
        h_est = [c for c in df_est.columns if c.lower() in ["h","horizon","time","t","year"]][0]
        
        col_est = None
        for c in df_est.columns:
            cl = c.lower().replace(" ","").replace(".","")
            target = rcp.lower().replace("rcp","").replace(".","")
            if target in cl:
                col_est = c
                break
        
        if col_est is None:
            num_cols = [c for c in df_est.select_dtypes(include=[np.number]).columns if c != h_est]
            idx = mapping.index((rcp, ssp))
            col_est = num_cols[idx]
        
        df_est = df_est[[h_est,col_est]].rename(columns={h_est:"H",col_est:"pIRF"})
        df_est = df_est[df_est["H"]<=100]
        
        plt.plot(
            df_est["H"], df_est["pIRF"],
            color="orange",
            linestyle="-",
            label=f"Watanabe and Cherubini (2026), {rcp}"
        )
        
        # Title only gas name
        plt.title(f"{gas}")
        plt.xlabel(r'Time since pulse ($\tau$)')
        #plt.xlabel("Time since pulse (years)")
        plt.ylabel("decay factor (prospective Impulse Response)")
        

        # Move legend for SSP585 panels to bottom right to avoid blocking orange curve
        #plt.legend(fontsize=12)
        if ssp == "ssp585" and gas == "CO2":
            plt.legend(fontsize=12, loc="lower right")
        else:
            plt.legend(fontsize=12)
            
                
        fname = f"{gas}_{rcp.replace('.','p')}.png"
        fpath = os.path.join(out_dir,fname)
        plt.savefig(fpath,dpi=300,bbox_inches="tight")
        plt.close()
        
        panel_paths.append(fpath)



In [5]:
# ---- 3x4 grid ----
ordered = []
for gas in ["CO2","CH4","N2O"]:
    for rcp,_ in mapping:
        ordered.append(os.path.join(out_dir,f"{gas}_{rcp.replace('.','p')}.png"))

imgs = [Image.open(p).convert("RGB") for p in ordered]

w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)

def pad(im,w,h):
    bg = Image.new("RGB",(w,h),(255,255,255))
    bg.paste(im,((w-im.size[0])//2,(h-im.size[1])//2))
    return bg

imgs = [pad(im,w,h) for im in imgs]

cols = 4
rows = 3
grid = Image.new("RGB",(w*cols,h*rows),(255,255,255))

for i,im in enumerate(imgs):
    r = i//cols
    c = i%cols
    grid.paste(im,(c*w,r*h))

final_path = os.path.join(out_dir, "pIRF_comp_whEST_addIPCC.png")
grid.save(final_path, "PNG")

print(f"Saved to: {final_path}")

Saved to: pIRF/plot_compare_Watanabe_AND_IPCC/pIRF_comp_whEST_addIPCC.png
